# Inference: YOLO11n on PDFs with GT Comparison

This notebook:

- Converts PDFs to images using pdf_to_image.py
- Loads best YOLO11n weights and runs inference
- Loads ground truth from selected_annotation.json
- Matches predictions to GT and reports per-class metrics


In [47]:
# Paths & configuration (updated for nested selected_annotations.json format)
from pathlib import Path
import os, json

ROOT = Path('/home/sara_team/Desktop/case')
BASE_DIR = ROOT / 'selected_output-20251115T200348Z-1-001'

# Primary (old) path (will be corrected if empty)
PDF_DIR = BASE_DIR / 'pdfs'

# Correct nested path containing pdfs/
NESTED_PDF_DIR = BASE_DIR / 'selected_output' / 'pdfs'
if not PDF_DIR.exists() or not any(PDF_DIR.glob('*.pdf')):
    PDF_DIR = NESTED_PDF_DIR

# Extra PDF directory to include in validation
EXTRA_PDF_DIR = ROOT / 'test-20251115T195726Z-1-001' / 'test'

ANNO_JSON = BASE_DIR / 'selected_output' / 'selected_annotations.json'
OUT_DIR = ROOT / 'runs' / 'inference' / 'yolo11n_eval'
IMAGES_DIR = OUT_DIR / 'images'
OUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# Class mapping (from data.yaml): 0:'qr', 1:'signature', 2:'stamp', 3:'stamp_q'
CLASS_MAP = {0: 'qr', 1: 'signature', 2: 'stamp', 3: 'stamp_q'}
CLASS_TO_ID = {v:k for k,v in CLASS_MAP.items()}
# Per-class confidence thresholds (qr=0.6 as requested)
CLASS_CONF = {0: 0.30, 1: 0.60, 2: 0.30, 3: 0.30}

print('PDF_DIR:', PDF_DIR)
print('EXTRA_PDF_DIR exists:', EXTRA_PDF_DIR.exists())
print('ANNO_JSON exists:', ANNO_JSON.exists())
print('OUT_DIR:', OUT_DIR)
base_count = len(list(PDF_DIR.glob('*.pdf')))
extra_count = len(list(EXTRA_PDF_DIR.glob('*.pdf'))) if EXTRA_PDF_DIR.exists() else 0
print('PDF count (base, extra, total):', base_count, extra_count, base_count+extra_count)

PDF_DIR: /home/sara_team/Desktop/case/selected_output-20251115T200348Z-1-001/selected_output/pdfs
EXTRA_PDF_DIR exists: True
ANNO_JSON exists: True
OUT_DIR: /home/sara_team/Desktop/case/runs/inference/yolo11n_eval
PDF count (base, extra, total): 45 13 58


In [ ]:
import sys, subprocess
def ensure(pkg):
    try:
        __import__(pkg)
        return True
    except Exception:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])
        return True

ensure('ultralytics')
ensure('pdf2image')

import torch
from ultralytics import YOLO
print('CUDA available:', torch.cuda.is_available(), '| device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))
DEVICE = '0' if torch.cuda.is_available() else 'cpu'

CUDA available: True | device count: 2
GPU 0: NVIDIA GeForce RTX 4090
GPU 1: NVIDIA GeForce RTX 4090


In [ ]:
from typing import List, Dict, Tuple
from pdf_to_image import convert_pdf_to_images


def convert_pdfs_in_dir(pdf_dir: Path, out_dir: Path, dpi: int = 200) -> Tuple[Dict[Tuple[str,int], Path], Dict[str,str]]:
    """
    Converts all PDFs under pdf_dir to PNG images into out_dir.
    Returns a mapping: (pdf_basename, page_idx) -> image_path.
    page_idx starts at 1.
    """
    mapping = {}
    base_to_filename = {}
    pdfs = sorted([p for p in pdf_dir.glob('**/*.pdf')])
    print(f'Found {len(pdfs)} PDFs in {pdf_dir}')
    for pdf in pdfs:
        base_to_filename[pdf.stem] = pdf.name
        created = convert_pdf_to_images(str(pdf), output_dir=str(out_dir), dpi=dpi)
        base = pdf.stem
        if not created:
            continue
        for img_path in created:
            imgp = Path(img_path)
            if imgp.stem == base:
                page_idx = 1
            else:
                page_idx = 1
                if '_page_' in imgp.stem:
                    try:
                        page_idx = int(imgp.stem.split('_page_')[-1])
                    except Exception:
                        page_idx = 1
            mapping[(base, page_idx)] = imgp
    print(f'Created {len(mapping)} page images from {pdf_dir}')
    return mapping, base_to_filename

def convert_pdfs_in_dirs(pdf_dirs: list[Path], out_dir: Path, dpi: int = 200):
    mapping_all: Dict[Tuple[str,int], Path] = {}
    base_to_filename_all: Dict[str,str] = {}
    total_pdfs = 0
    for d in pdf_dirs:
        if not d or not Path(d).exists():
            continue
        mapping, base_to_filename = convert_pdfs_in_dir(Path(d), out_dir, dpi=dpi)
        mapping_all.update(mapping)
        base_to_filename_all.update(base_to_filename)
        total_pdfs += len(list(Path(d).glob('**/*.pdf')))
    print(f'Total PDFs processed across dirs: {total_pdfs} -> pages: {len(mapping_all)}')
    return mapping_all, base_to_filename_all

In [ ]:
from typing import Any

def parse_selected_annotations_nested(json_path: Path) -> tuple[Dict[Tuple[str,int], list], Dict[Tuple[str,int], tuple[int,int]]]:
    with open(json_path, 'r', encoding='utf-8') as f: data = json.load(f)
    gt = {}; page_sizes = {}
    for pdf_name, pages in data.items():
        base = Path(pdf_name).stem
        for page_key, page_obj in pages.items():
            if not page_key.startswith('page_'): continue
            try: 
                page_idx = int(page_key.split('_')[1]) 
            except Exception: 
                page_idx = 1
            ps = page_obj.get('page_size', {})
            pw, ph = ps.get('width'), ps.get('height')
            if isinstance(pw, (int,float)) and isinstance(ph, (int,float)):
                page_sizes[(base, page_idx)] = (int(pw), int(ph))
            ann_list = page_obj.get('annotations', [])
            for ann_entry in ann_list:
                if not isinstance(ann_entry, dict): continue
                annotation_id, details = next(iter(ann_entry.items())) if ann_entry else (None, None)
                if not details: continue
                cat = details.get('category')
                bbox_obj = details.get('bbox', {})
                x = bbox_obj.get('x'); y = bbox_obj.get('y'); w = bbox_obj.get('width'); h = bbox_obj.get('height')
                if None in (x,y,w,h): continue
                cls_id = CLASS_TO_ID.get(str(cat).strip().lower())
                if cls_id is None: continue
                xyxy = [float(x), float(y), float(x+w), float(y+h)]
                gt.setdefault((base, page_idx), []).append({'cls': cls_id, 'xyxy': xyxy})
    return gt, page_sizes
GT_RAW, PAGE_SIZES = parse_selected_annotations_nested(ANNO_JSON)
print('GT pages (raw):', len(GT_RAW))
for k,v in list(GT_RAW.items())[:3]: print(k, '->', v[:2], '...')
print('Sample page sizes:', list(PAGE_SIZES.items())[:3])

GT pages (raw): 91
('локалсмета-', 3) -> [{'cls': 1, 'xyxy': [510.0, 146.0, 760.0, 244.89]}, {'cls': 1, 'xyxy': [607.0, 232.0, 899.13, 316.2]}] ...
('Разрешназемлю-41-', 1) -> [{'cls': 2, 'xyxy': [709.0, 1184.0, 917.76, 1402.1100000000001]}, {'cls': 1, 'xyxy': [702.0, 1227.0, 887.61, 1367.66]}] ...
('письмо-11', 2) -> [{'cls': 2, 'xyxy': [613.0, 321.0, 846.366, 560.145]}, {'cls': 1, 'xyxy': [582.0, 312.0, 682.01, 377.83]}] ...
Sample page sizes: [(('локалсмета-', 3), (1684, 1190)), (('Разрешназемлю-41-', 1), (1190, 1684)), (('письмо-11', 2), (1190, 1684))]


In [ ]:
import cv2
def scale_gt_to_images(gt_map, page_sizes, page_image_map):
    scaled = {}
    stats = {'scaled':0,'pages_missing_image':0,'pages_missing_size':0,'orientation_swaps':0}
    for key, boxes in gt_map.items():
        base, page_idx = key
        img_path = page_image_map.get((base, page_idx))
        if not img_path or not Path(img_path).exists():
            stats['pages_missing_image'] += 1
            continue
        img = cv2.imread(str(img_path))
        if img is None: stats['pages_missing_image'] += 1; continue
        h_img, w_img = img.shape[:2]
        ps = page_sizes.get(key)
        if not ps: stats['pages_missing_size'] += 1; scaled[key] = boxes; continue
        pw, ph = ps
        sx, sy = (w_img / pw, h_img / ph) if pw>0 and ph>0 else (1.0,1.0)
        def in_bounds(bx):
            x1,y1,x2,y2 = bx
            return 0 <= x1 < w_img and 0 <= y1 < h_img and 0 < x2 <= w_img+2 and 0 < y2 <= h_img+2
        scaled_boxes_direct = []
        for b in boxes:
            x1,y1,x2,y2 = b['xyxy']; scaled_boxes_direct.append({'cls': b['cls'], 'xyxy': [x1*sx, y1*sy, x2*sx, y2*sy]})
        if all(in_bounds(b['xyxy']) for b in scaled_boxes_direct):
            scaled[key] = scaled_boxes_direct; stats['scaled'] += 1; continue

        sx2, sy2 = (w_img / ph, h_img / pw) if ph>0 and pw>0 else (1.0,1.0)
        scaled_boxes_swapped = []
        for b in boxes:
            x1,y1,x2,y2 = b['xyxy']; scaled_boxes_swapped.append({'cls': b['cls'], 'xyxy': [x1*sx2, y1*sy2, x2*sx2, y2*sy2]})
        if all(in_bounds(b['xyxy']) for b in scaled_boxes_swapped):
            scaled[key] = scaled_boxes_swapped; stats['orientation_swaps'] += 1; continue
        scaled[key] = boxes
    return scaled, stats
print('Defined scale_gt_to_images(). Compute after images are rendered.')

Defined scale_gt_to_images(). Compute after images are rendered.


In [ ]:
# Prediction: run YOLO on rendered images
import math

def infer_images(model: YOLO, page_map: Dict[tuple, Path], class_conf: dict[int, float], default_conf: float = 0.30):
    preds: Dict[tuple, list] = {}
    base_conf = min(class_conf.values()) if class_conf else default_conf
    for (pdf_base, page_idx), img_path in page_map.items():
        res = model.predict(source=str(img_path), imgsz=640, conf=base_conf, device=DEVICE, verbose=False)
        if not res:
            continue
        r = res[0]
        dets = []
        if r.boxes is not None and len(r.boxes) > 0:
            xyxy = r.boxes.xyxy.cpu().numpy().tolist()
            cls = r.boxes.cls.cpu().numpy().astype(int).tolist()
            confs = r.boxes.conf.cpu().numpy().tolist()
            for b, c, s in zip(xyxy, cls, confs):
                thr = class_conf.get(int(c), default_conf)
                if float(s) < float(thr):
                    continue
                dets.append({'cls': int(c), 'xyxy': [float(x) for x in b], 'conf': float(s)})
        preds[(pdf_base, page_idx)] = dets
    print('Predicted pages:', len(preds))
    return preds

PAGE_MAP, BASE_TO_FILENAME = convert_pdfs_in_dirs([PDF_DIR, EXTRA_PDF_DIR], IMAGES_DIR, dpi=200)
# Load model and run inference
model = YOLO('/home/sara_team/Desktop/case/runs/train/yolo11n_idp_qr3/weights/best.pt')
PRED = infer_images(model, PAGE_MAP, class_conf=CLASS_CONF, default_conf=0.30)

Found 45 PDFs in /home/sara_team/Desktop/case/selected_output-20251115T200348Z-1-001/selected_output/pdfs
Converting АПЗ-.pdf to images...
Saved: АПЗ-_page_001.png
Saved: АПЗ-_page_001.png
Saved: АПЗ-_page_002.png
Saved: АПЗ-_page_002.png
Saved: АПЗ-_page_003.png
Saved: АПЗ-_page_003.png
Saved: АПЗ-_page_004.png
Saved: АПЗ-_page_004.png
Saved: АПЗ-_page_005.png
Saved: АПЗ-_page_005.png
Saved: АПЗ-_page_006.png
Saved: АПЗ-_page_006.png
Saved: АПЗ-_page_007.png
Saved: АПЗ-_page_007.png
Saved: АПЗ-_page_008.png
Saved: АПЗ-_page_008.png
Saved: АПЗ-_page_009.png
Converting АПЗ-2.pdf to images...
Saved: АПЗ-_page_009.png
Converting АПЗ-2.pdf to images...
Saved: АПЗ-2_page_001.png
Saved: АПЗ-2_page_001.png
Saved: АПЗ-2_page_002.png
Saved: АПЗ-2_page_002.png
Saved: АПЗ-2_page_003.png
Saved: АПЗ-2_page_003.png
Saved: АПЗ-2_page_004.png
Saved: АПЗ-2_page_004.png
Saved: АПЗ-2_page_005.png
Saved: АПЗ-2_page_005.png
Saved: АПЗ-2_page_006.png
Saved: АПЗ-2_page_006.png
Saved: АПЗ-2_page_007.png
Saved

In [54]:
# Compute GT scaling AFTER images are rendered
GT_SCALED, SCALE_STATS = scale_gt_to_images(GT_RAW, PAGE_SIZES, PAGE_MAP)
print('Scaling stats:', SCALE_STATS)
for k,v in list(GT_SCALED.items())[:2]: print('Scaled sample', k, v[:1])

Scaling stats: {'scaled': 91, 'pages_missing_image': 0, 'pages_missing_size': 0, 'orientation_swaps': 0}
Scaled sample ('локалсмета-', 3) [{'cls': 1, 'xyxy': [1416.7339667458432, 405.6100840336134, 2111.211401425178, 680.3414621848739]}]
Scaled sample ('Разрешназемлю-41-', 1) [{'cls': 2, 'xyxy': [1969.709243697479, 3289.04513064133, 2549.676100840336, 3894.9350237529693]}]


In [ ]:
import numpy as np
from collections import defaultdict

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    inter_x1, inter_y1 = max(ax1,bx1), max(ay1,by1)
    inter_x2, inter_y2 = min(ax2,bx2), min(ay2,by2)
    iw, ih = max(0.0, inter_x2-inter_x1), max(0.0, inter_y2-inter_y1)
    inter = iw*ih
    area_a = max(0.0, (ax2-ax1)) * max(0.0, (ay2-ay1))
    area_b = max(0.0, (bx2-bx1)) * max(0.0, (by2-by1))
    union = area_a + area_b - inter + 1e-6
    return inter/union

def match_and_score(gt_map, pred_map, iou_thr=0.5):
    TP = defaultdict(int); FP = defaultdict(int); FN = defaultdict(int)
    keys = set(gt_map.keys())
    for key in keys:
        gts = gt_map.get(key, []); prs = pred_map.get(key, []); used_pr = set();
        for g in gts:
            gc = g['cls']; gbox = g['xyxy']; best, best_i = 0.0, -1;
            for pi, p in enumerate(prs):
                if pi in used_pr or int(p['cls']) != int(gc): continue
                i = iou_xyxy(gbox, p['xyxy']);
                if i > best: best, best_i = i, pi
            if best >= iou_thr and best_i >= 0:
                TP[gc]+=1; used_pr.add(best_i)
            else: FN[gc]+=1
        for pi,p in enumerate(prs):
            if pi in used_pr: continue
            FP[int(p['cls'])]+=1
    metrics=[]
    for cid,cname in CLASS_MAP.items():
        tp,fp,fn = TP[cid],FP[cid],FN[cid]
        prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
        rec = tp/(tp+fn) if (tp+fn)>0 else 0.0
        f1 = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
        metrics.append({'class_id':cid,'class_name':cname,'tp':tp,'fp':fp,'fn':fn,'precision':round(prec,4),'recall':round(rec,4),'f1':round(f1,4)})
    return metrics

METRICS = match_and_score(GT_SCALED, PRED, iou_thr=0.5)
for row in METRICS: print(row)
with open(OUT_DIR / 'metrics.json', 'w', encoding='utf-8') as f: json.dump(METRICS, f, ensure_ascii=False, indent=2)
print('Saved (scaled GT) metrics to:', OUT_DIR / 'metrics.json')

{'class_id': 0, 'class_name': 'qr', 'tp': 78, 'fp': 1, 'fn': 17, 'precision': 0.9873, 'recall': 0.8211, 'f1': 0.8966}
{'class_id': 1, 'class_name': 'signature', 'tp': 63, 'fp': 9, 'fn': 40, 'precision': 0.875, 'recall': 0.6117, 'f1': 0.72}
{'class_id': 2, 'class_name': 'stamp', 'tp': 58, 'fp': 2, 'fn': 2, 'precision': 0.9667, 'recall': 0.9667, 'f1': 0.9667}
{'class_id': 3, 'class_name': 'stamp_q', 'tp': 0, 'fp': 0, 'fn': 0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
Saved (scaled GT) metrics to: /home/sara_team/Desktop/case/runs/inference/yolo11n_eval/metrics.json


In [ ]:
import cv2
def _in_bounds_xyxy(bx, w, h):
    x1,y1,x2,y2 = bx
    return (0 <= x1 <= w and 0 <= y1 <= h and 0 <= x2 <= w and 0 <= y2 <= h and x2>x1 and y2>y1)

def _scale_pred_xyxy_to_pagesize(xyxy, img_wh, target_wh):
    (w_img, h_img) = img_wh; (w_t, h_t) = target_wh
    if w_img<=0 or h_img<=0 or w_t<=0 or h_t<=0: return xyxy
    sx, sy = w_t / w_img, h_t / h_img
    x1,y1,x2,y2 = xyxy
    cand1 = [x1*sx, y1*sy, x2*sx, y2*sy]
    if _in_bounds_xyxy(cand1, w_t+2, h_t+2):
        return cand1
    sx2, sy2 = w_t / h_img, h_t / w_img
    cand2 = [x1*sx2, y1*sy2, x2*sx2, y2*sy2]
    if _in_bounds_xyxy(cand2, w_t+2, h_t+2):
        return cand2
    return cand1  # fallback

def save_predictions_hier(pred_map, page_map, base_to_filename, page_sizes, out_path: Path):
    output = {}
    for (base, page_idx), dets in sorted(pred_map.items(), key=lambda kv: (kv[0][0], kv[0][1])):
        if not dets:
        pdf_name = base_to_filename.get(base, base + '.pdf')
        img_path = page_map.get((base, page_idx))
        if not img_path or not Path(img_path).exists():
            continue
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h_img, w_img = img.shape[:2]
        tgt_size = page_sizes.get((base, page_idx))
        if not tgt_size:
            tgt_size = (w_img, h_img)
        w_t, h_t = int(tgt_size[0]), int(tgt_size[1])
        scaled = []
        for d in dets:
            x1,y1,x2,y2 = d['xyxy']
            sx1,sy1,sx2,sy2 = _scale_pred_xyxy_to_pagesize([x1,y1,x2,y2], (w_img,h_img), (w_t,h_t))
            w_box, h_box = max(0.0, sx2 - sx1), max(0.0, sy2 - sy1)
            if w_box <= 0 or h_box <= 0:
                continue
            scaled.append({
                'category': CLASS_MAP.get(int(d['cls']), str(d['cls'])),
                'bbox': {'x': float(sx1), 'y': float(sy1), 'width': float(w_box), 'height': float(h_box)},
                'area': float(w_box * h_box)
            })
        if not scaled:
            continue
        page_key = f'page_{int(page_idx)}'
        if pdf_name not in output:
            output[pdf_name] = {}
        page_entry = {'annotations': [], 'page_size': {'width': w_t, 'height': h_t}}
        for i, obj in enumerate(scaled, start=1):
            ann_id = f'annotation_{i}'
            page_entry['annotations'].append({ann_id: obj})
        output[pdf_name][page_key] = page_entry
    output = {k:v for k,v in output.items() if any(v.values())}
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    return out_path

PRED_HIER_PATH = save_predictions_hier(PRED, PAGE_MAP, BASE_TO_FILENAME, PAGE_SIZES, OUT_DIR / 'predictions_all.json')
print('Saved hierarchical predictions to:', PRED_HIER_PATH)

Saved hierarchical predictions to: /home/sara_team/Desktop/case/runs/inference/yolo11n_eval/predictions_all.json
